In [1]:
import pandas as pd

## cargando dataframes necesariso para la pregunta

In [2]:
order_items_dataset = pd.read_csv('../data/order_items_dataset.csv')
print(order_items_dataset.shape)
order_items_dataset.head(1)

(112650, 7)


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.9,13.29


In [3]:
products_dataset= pd.read_csv('../data/products_dataset.csv')
print(products_dataset.shape)
products_dataset.head(1)

(32951, 9)


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0


In [4]:
product_category_name_translation= pd.read_csv('../data/product_category_name_translation.csv')
print(product_category_name_translation.shape)
product_category_name_translation.head(1)

(71, 2)


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty


In [5]:
orders_dataset = pd.read_csv('../data/orders_dataset.csv')
print(orders_dataset.shape)
orders_dataset.head(1)

(99441, 8)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00


## EDA sencillo

1. ¿Cuáles son los 5 productos (o categorías de productos) más vendidos en términos de 
volumen y cuáles en términos de ingresos totales?

In [6]:
orders_dataset['order_status'].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

Depende del objetivo del informe puede cambiar el valor, esto debido a la columna 'order_status' del orders_dataset, ya que no todos los pedidos han sido entregados, pero pensando en un informe financiero sólo se va a a tomar como producto vendido aquellos productos con estado 'delivered' (entregado).

In [7]:
orders_delivered = orders_dataset[orders_dataset['order_status'] == 'delivered'].copy()

orders_delivered.head(2)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00


In [8]:
print(orders_delivered['order_id'].dtype)
print(order_items_dataset['order_id'].dtype)

str
str


In [9]:
#para saber producto vendido, vendedor y precio
merged_df = orders_delivered.merge(order_items_dataset, on='order_id', how='inner')

merged_df.head(1)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,1,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72


In [10]:
print(merged_df['product_id'].dtype)
print(products_dataset['product_id'].dtype)

str
str


In [11]:
# para traer la categoría
merged_df = merged_df.merge(products_dataset, on='product_id', how='inner')

merged_df.head(1)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,...,price,freight_value,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,1,87285b34884572647811a353c7ac498a,...,29.99,8.72,utilidades_domesticas,40.0,268.0,4.0,500.0,19.0,8.0,13.0


In [12]:
# para saber el nombre del producto en ingles
merged_df = merged_df.merge(product_category_name_translation, on='product_category_name', how='left')

merged_df.head(1)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,...,freight_value,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,1,87285b34884572647811a353c7ac498a,...,8.72,utilidades_domesticas,40.0,268.0,4.0,500.0,19.0,8.0,13.0,housewares


## explorando el dataframe resultante

In [13]:
merged_df.shape

(110197, 23)

In [14]:
merged_df.dtypes

order_id                             str
customer_id                          str
order_status                         str
order_purchase_timestamp             str
order_approved_at                    str
order_delivered_carrier_date         str
order_delivered_customer_date        str
order_estimated_delivery_date        str
order_item_id                      int64
product_id                           str
seller_id                            str
shipping_limit_date                  str
price                            float64
freight_value                    float64
product_category_name                str
product_name_lenght              float64
product_description_lenght       float64
product_photos_qty               float64
product_weight_g                 float64
product_length_cm                float64
product_height_cm                float64
product_width_cm                 float64
product_category_name_english        str
dtype: object

In [15]:
merged_df.isnull().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                  15
order_delivered_carrier_date        2
order_delivered_customer_date       8
order_estimated_delivery_date       0
order_item_id                       0
product_id                          0
seller_id                           0
shipping_limit_date                 0
price                               0
freight_value                       0
product_category_name            1537
product_name_lenght              1537
product_description_lenght       1537
product_photos_qty               1537
product_weight_g                   18
product_length_cm                  18
product_height_cm                  18
product_width_cm                   18
product_category_name_english    1559
dtype: int64

In [16]:
#manejo de datos nulos para los productos sin categoría y nombre final a usar
merged_df['categoria_producto_final'] = merged_df['product_category_name_english'].fillna(merged_df['product_category_name'])
merged_df['categoria_producto_final'] = merged_df['categoria_producto_final'].fillna('unknown_category')

merged_df.head(1)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,...,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,categoria_producto_final
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,1,87285b34884572647811a353c7ac498a,...,utilidades_domesticas,40.0,268.0,4.0,500.0,19.0,8.0,13.0,housewares,housewares


In [27]:
# top 5 productos por volumen
top_5_volume = (
    merged_df.groupby('categoria_producto_final')
    .size()
    .reset_index(name='volumen_total')
    .sort_values(by='volumen_total', ascending=False)
    .head(5)
)
top_5_volume


,categoria_producto_final,volumen_total
7,bed_bath_table,10953
43,health_beauty,9465
67,sports_leisure,8431
39,furniture_decor,8160
15,computers_accessories,7644


In [20]:
# top 5 productos por ingresos
top_5_revenue = (
    merged_df.groupby('categoria_producto_final')['price']
    .sum()
    .reset_index(name='ingreso_total')
    .sort_values(by='ingreso_total', ascending=False)
    .head(5)
)

top_5_revenue

,categoria_producto_final,ingreso_total
43,health_beauty,1233131.72
73,watches_gifts,1166176.98
7,bed_bath_table,1023434.76
67,sports_leisure,954852.55
15,computers_accessories,888724.61
